# CineMind AI - Colab Quickstart
Run this notebook in Google Colab to download MovieLens, train the model, and export artifacts.

In [ ]:
!pip -q install pandas numpy scikit-learn scipy joblib matplotlib seaborn plotly

In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -q ml-latest-small.zip

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import joblib

movies = pd.read_csv('ml-latest-small/movies.csv')
ratings = pd.read_csv('ml-latest-small/ratings.csv')
data = ratings.merge(movies, on='movieId', how='left')
user_movie = data.pivot_table(index='userId', columns='title', values='rating').fillna(0)
movie_user = user_movie.T

user_model = NearestNeighbors(metric='cosine', algorithm='brute')
user_model.fit(user_movie)
user_dist, user_idx = user_model.kneighbors(user_movie, n_neighbors=min(51, user_movie.shape[0]))
user_scores = 1 - user_dist

movie_model = NearestNeighbors(metric='cosine', algorithm='brute')
movie_model.fit(movie_user)
movie_dist, movie_idx = movie_model.kneighbors(movie_user, n_neighbors=min(51, movie_user.shape[0]))
movie_scores = 1 - movie_dist

scaler = StandardScaler()
scaled = scaler.fit_transform(user_movie)
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
clusters = kmeans.fit_predict(scaled)

pca = PCA(n_components=2, random_state=42)
pca_embeddings = pca.fit_transform(scaled)

popularity = data.groupby('title').agg(rating_count=('rating','count'), rating_mean=('rating','mean')).sort_values(['rating_count','rating_mean'], ascending=False).reset_index()
user_activity = data.groupby('userId').agg(ratings_count=('rating','count'), rating_mean=('rating','mean')).reset_index()

artifacts = {
    'user_movie_matrix': user_movie,
    'movie_user_matrix': movie_user,
    'user_neighbor_indices': user_idx,
    'user_neighbor_scores': user_scores,
    'movie_neighbor_indices': movie_idx,
    'movie_neighbor_scores': movie_scores,
    'user_clusters': clusters,
    'popularity': popularity,
    'user_activity': user_activity,
    'pca_embeddings': pca_embeddings,
}

joblib.dump(artifacts, 'artifacts.joblib')
print('Saved artifacts.joblib')

Upload artifacts.joblib into the project's model/ folder to run the Streamlit app.